# 🚜 FarmifAI: Evaluación Comparativa de Sistemas de Recomendación Basados en Conocimiento
### Benchmarking Riguroso Frente a Baselines Tradicionales: AIML (Graphmaster) vs. Okapi BM25 (τ) vs. FAQ Finder (TCBR/WordNet) vs. Sistema Experto Rete

---

## 🎯 Objetivo de este Notebook en el Marco de la Tesis
Para validar con rigor académico el desarrollo de un sistema de recomendación generativo basado en **RAG (Recuperador Híbrido + Reranker + SLM)** en la tesis *FarmifAI*, es metodológicamente indispensable contrastarlo contra los enfoques canónicos de la literatura de **Sistemas de Recomendación Basados en Conocimiento (Knowledge-Based Recommenders - KBR)** y recuperación de respuestas sobre repositorios cerrados (*Textual Case-Based Reasoning* / Selección de Respuestas Prehechas).

### 🏛️ Los 4 Paradigmas de Recomendación Tradicionales Evaluados

1. **Motores Sintácticos y Gramáticas (Extremo Simbólico Determinista):**
   - **Método:** *AIML (Artificial Intelligence Markup Language)* con emparejamiento sobre árbol de prefijos (*Graphmaster Pattern Matching*).
   - **Referencia Canónica:** Wallace, R. S. (2009). *The Anatomy of A.L.I.C.E.* Springer.
   - **Fundamento:** Selección por coincidencia exacta y comodines sintácticos (`_`, `*`) en un árbol de discriminación de tokens compilado a partir de los patrones de entrenamiento.
2. **Modelos Probabilísticos y Vectoriales (Information Retrieval sobre Respuestas):**
   - **Método:** *Okapi BM25 con función de corte $\tau$* y mecanismo de *fallback*.
   - **Referencia Canónica:** Robertson, S., & Zaragoza, H. (2009). *The Probabilistic Relevance Framework: BM25 and Beyond*. Foundations and Trends in Information Retrieval.
   - **Fundamento:** Modela la saturación de términos ($k_1$) y la normalización por longitud de respuesta ($b$). Evalúa la consulta contra cada respuesta candidata $a_i$. Si $\max(\text{Score}_{\text{BM25}}(q, a_i)) < \tau$, emite respuesta de *fallback*.
3. **Razonamiento Basado en Casos Textuales - TCBR (Enfoque Multicanal):**
   - **Método:** *FAQ Finder (Burke et al.)* adaptado a selección de respuestas mediante similitud léxica vectorial (TF-IDF) y distancia semántica ontológica sobre el grafo de *WordNet*.
   - **Referencia Canónica:** Burke, R. D., Hammond, K. J., Kulyukin, V., Lytinen, S. L., Tomuro, N., & Schoenberg, S. (1997). *Question Answering from Local Library Resources: An Investigation of FAQ Finder*. User Modeling and User-Adapted Interaction.
   - **Fundamento:** Resuelve la brecha de vocabulario combinando coincidencia léxica con similitud conceptual ontológica (Wu-Palmer sobre WordNet en español) en dos etapas (filtro rápido Top-20 y reordenamiento semántico).
4. **Motores de Reglas y Sistemas Expertos (Inferencia Lógica Pura):**
   - **Método:** *Sistema Experto Basado en Reglas con Algoritmo Rete* (*Forward Chaining*).
   - **Referencia Canónica:** Forgy, C. L. (1982). *Rete: A Fast Algorithm for the Many Pattern/Many Object Pattern Match Problem*. Artificial Intelligence; y Felfernig, A., et al. (2006). *Constraint-based recommender systems*.
   - **Fundamento:** La consulta se descompone en hechos sintácticos agrícolas (`WorkingMemory`: cultivo, labor, patógeno/síntoma), propagándose por una red de nodos alfa y beta para disparar la recomendación experta con resolución de conflictos por especificidad.


## 1. Configuración del Entorno e Instalación de Dependencias

Instalamos las librerías necesarias para el procesamiento de lenguaje natural en español, los evaluadores IR, el grafo WordNet multilingüe, y el evaluador BERTScore acelerado por GPU CUDA.


In [ ]:
%%capture
import os, sys, time, json, re, math, random
from collections import defaultdict, namedtuple
from typing import List, Dict, Any, Tuple, Optional

print("[INFO] Actualizando gestor de paquetes uv...")
!pip install --upgrade -qqq uv

print("[INFO] Instalando librerías especializadas (rank-bm25, bert-score, nltk, scikit-learn, torch)...")
!uv pip install -qqq rank-bm25 bert-score scikit-learn nltk pandas matplotlib seaborn tqdm torch

import torch
import nltk
print("[INFO] Descargando recursos lingüísticos de NLTK en español...")
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("[OK] Todas las dependencias se han instalado y configurado correctamente.")


In [ ]:
import torch
import psutil

print("=" * 65)
print(" 🖥️ DIAGNÓSTICO DEL ENTORNO DE EVALUACIÓN (GOOGLE COLAB)")
print("=" * 65)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Detectada: {gpu_name}")
    print(
        f"📊 VRAM Disponible: {vram_total:.2f} GB (Aceleración de BERTScore activada)")
    print(f"🚀 CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ ADVERTENCIA: No se detectó GPU CUDA. BERTScore se ejecutará en CPU (más lento).")
    print("Sugerencia: 'Entorno de ejecución' -> 'Cambiar tipo de entorno de ejecución' -> 'T4 GPU'.")

ram_total = psutil.virtual_memory().total / (1024**3)
print(f"💾 RAM del Sistema: {ram_total:.2f} GB")
print("=" * 65)

## 2. Carga Defensiva y Validación de Datasets

El sistema requiere dos conjuntos en formato JSON Lines (`.jsonl`):
1. **`dataset_agricola_train.jsonl`**: Base de conocimiento de entrenamiento (~7,891 registros). De aquí se extrae exclusivamente el catálogo de respuestas prehechas candidatas.
2. **`dataset_agricola_eval.jsonl`**: Conjunto de evaluación held-out (~876 preguntas) sin solapamiento con el entrenamiento.

Si alguno de los dos archivos no existe en el entorno local de Colab, se solicitará automáticamente su subida interactiva.


In [ ]:
import os
import json
import re
from google.colab import files as colab_files


def verify_or_upload_file(filename: str, description: str) -> str:
    """Verifica si el archivo existe en Colab; si no, solicita al usuario subirlo interactivamente."""
    if os.path.exists(filename):
        print(f"✅ Archivo encontrado: '{filename}' ({description})")
        return filename

    print(f"\n⚠️ Archivo no encontrado: '{filename}'")
    print(
        f"👉 Por favor, selecciona y sube tu archivo '{filename}' ({description}):")
    uploaded = colab_files.upload()

    if not uploaded:
        raise FileNotFoundError(
            f"❌ No se subió ningún archivo. Se requiere '{filename}' para continuar.")

    uploaded_name = list(uploaded.keys())[0]
    if uploaded_name != filename and os.path.exists(uploaded_name):
        os.rename(uploaded_name, filename)
    print(f"✅ Archivo subido y listo: '{filename}'")
    return filename


TRAIN_DATASET_FILE = verify_or_upload_file(
    "dataset_agricola_train.jsonl", "Base de Conocimiento de Respuestas (Train)")
EVAL_DATASET_FILE = verify_or_upload_file(
    "dataset_agricola_eval.jsonl", "Dataset de Evaluación (Eval)")

In [ ]:
# --- FUNCIONES DE EXTRACCIÓN Y LIMPIEZA DE MUESTRAS ---

def extract_qa_pair(sample: dict, idx: int) -> Optional[dict]:
    """Extrae pregunta limpia, contexto knowledge y respuesta final de un registro JSONL."""
    messages = sample.get("messages", [])
    user_msg = next((m["content"]
                    for m in messages if m.get("role") == "user"), "")
    assistant_msg = next((m["content"]
                         for m in messages if m.get("role") == "assistant"), "")

    if not user_msg or not assistant_msg:
        return None

    # Extraer knowledge si está presente
    k_match = re.search(r'<knowledge>(.*?)</knowledge>', user_msg, re.DOTALL)
    knowledge = k_match.group(1).strip() if k_match else ""

    # Limpiar pregunta del usuario eliminando bloque <knowledge>
    question = re.sub(r'<knowledge>.*?</knowledge>', '',
                      user_msg, flags=re.DOTALL).strip()
    if not question:
        question = user_msg.strip()

    # Extraer respuesta final eliminando bloque <reasoning>
    ans_match = re.search(r'<answer>(.*?)</answer>', assistant_msg, re.DOTALL)
    if ans_match:
        answer = ans_match.group(1).strip()
    else:
        answer = re.sub(r'<reasoning>.*?</reasoning>', '',
                        assistant_msg, flags=re.DOTALL).strip()
        answer = answer.replace('<answer>', '').replace(
            '</answer>', '').strip()

    return {
        "id": idx,
        "question": question,
        "knowledge": knowledge,
        "answer": answer
    }


print("[INFO] Cargando y procesando base de conocimiento candidata (dataset_agricola_train.jsonl)...")
candidate_corpus = []
train_answers_set = set()

with open(TRAIN_DATASET_FILE, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        line = line.strip()
        if line:
            try:
                sample = json.loads(line)
                qa = extract_qa_pair(sample, i)
                if qa and qa["answer"] and qa["answer"] not in train_answers_set:
                    train_answers_set.add(qa["answer"])
                    candidate_corpus.append(qa)
            except Exception:
                continue

print(
    f"✅ Base de Respuestas Prehechas cargada: {len(candidate_corpus):,} respuestas candidatas únicas indexables.")

print("\n[INFO] Cargando y procesando dataset de evaluación held-out (dataset_agricola_eval.jsonl)...")
eval_dataset = []
with open(EVAL_DATASET_FILE, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        line = line.strip()
        if line:
            try:
                sample = json.loads(line)
                qa = extract_qa_pair(sample, i)
                if qa and qa["question"] and qa["answer"]:
                    eval_dataset.append(qa)
            except Exception:
                continue

print(
    f"✅ Dataset de Evaluación cargado: {len(eval_dataset):,} preguntas de prueba.")

# Verificación de No-Fuga de Conocimiento (Zero Leakage Check)
train_q_set = {c["question"].strip().lower() for c in candidate_corpus}
leakage_count = sum(
    1 for e in eval_dataset if e["question"].strip().lower() in train_q_set)
print(
    f"🛡️ Verificación de No-Fuga: {leakage_count} preguntas de evaluación se encuentran en el conjunto de entrenamiento (0.0% = Perfecto).")

print(f"\n🔍 Ejemplo de Muestra de Evaluación [Índice 0]:")
print(f"  Pregunta: {eval_dataset[0]['question'][:110]}...")
print(f"  Respuesta Referencia: {eval_dataset[0]['answer'][:110]}...")

## 3. Implementación de los 4 Sistemas de Recomendación Basados en Conocimiento

A continuación se implementan los 4 sistemas de recomendación orientados a **seleccionar la respuesta más relevante** del catálogo cerrado ante la pregunta del agricultor.

---

### 3.1. Método 1: Motores Sintácticos y Gramáticas — AIML (Graphmaster Pattern Matching)
- **Referencia Clave:** Wallace, R. S. (2009). *The Anatomy of A.L.I.C.E.* Springer.
- **Diseño Algorítmico:**
  - El *Graphmaster* es un árbol de prefijos de tokens (*trie* bidireccional) donde cada nodo almacena palabras o comodines sintácticos.
  - Orden canónico de precedencia (Wallace, 2009):
    1. `_` (comodín prioritario, coincide con una o más palabras)
    2. Palabra léxica exacta (coincidencia literal del token normalizado)
    3. `*` (comodín estándar, coincide con una o más palabras)
  - Las reglas y plantillas sintácticas se compilan **únicamente a partir del conjunto de entrenamiento**. Si una consulta de prueba no activa ninguna ruta válida, se dispara la plantilla por defecto (*fallback*).


In [ ]:
# --- IMPLEMENTACIÓN PURA Y OPTIMIZADA DE AIML GRAPHMASTER (WALLACE, 2009) ---

class GraphmasterNode:
    def __init__(self, token: str = ""):
        self.token = token
        self.children: Dict[str, 'GraphmasterNode'] = {}
        self.template: Optional[str] = None
        self.is_terminal: bool = False


class AIMLGraphmaster:
    """
    Implementación en Python puro de la estructura Graphmaster de AIML.
    Implementa backtracking defensivo respetando la precedencia canónica: '_' > Palabra Exacta > '*'.
    """

    def __init__(self, fallback_response: str = "No se encontró coincidencia en el motor de patrones sintácticos AIML."):
        self.root = GraphmasterNode("<ROOT>")
        self.fallback_response = fallback_response
        self.total_patterns = 0

    @staticmethod
    def normalize_tokens(text: str) -> List[str]:
        """Normaliza texto a tokens en mayúsculas sin puntuación para AIML."""
        text_clean = re.sub(r'[^a-zA-Z0-9áéíóúÁÉÍÓÚñÑüÜ\s]', ' ', text.upper())
        tokens = [t.strip() for t in text_clean.split() if t.strip()]
        return tokens

    def add_category(self, pattern_tokens: List[str], template_answer: str):
        """Inserta una lista de tokens de patrón y su plantilla de respuesta en el Graphmaster."""
        if not pattern_tokens:
            return

        current = self.root
        for token in pattern_tokens:
            if token not in current.children:
                current.children[token] = GraphmasterNode(token)
            current = current.children[token]
        current.is_terminal = True
        current.template = template_answer
        self.total_patterns += 1

    def _match_recursive(self, node: GraphmasterNode, tokens: List[str], token_idx: int) -> Optional[str]:
        """Búsqueda recursiva con backtracking y precedencia: '_' -> Token Exacto -> '*'."""
        if token_idx == len(tokens):
            if node.is_terminal and node.template is not None:
                return node.template
            if "*" in node.children and node.children["*"].is_terminal:
                return node.children["*"].template
            if "_" in node.children and node.children["_"].is_terminal:
                return node.children["_"].template
            return None

        current_token = tokens[token_idx]

        # 1. Precedencia 1: Comodín Prioritario '_'
        if "_" in node.children:
            wildcard_node = node.children["_"]
            for next_idx in range(token_idx + 1, len(tokens) + 1):
                res = self._match_recursive(wildcard_node, tokens, next_idx)
                if res is not None:
                    return res

        # 2. Precedencia 2: Coincidencia Exacta de Palabra
        if current_token in node.children:
            res = self._match_recursive(
                node.children[current_token], tokens, token_idx + 1)
            if res is not None:
                return res

        # 3. Precedencia 3: Comodín Estándar '*'
        if "*" in node.children:
            wildcard_node = node.children["*"]
            for next_idx in range(token_idx + 1, len(tokens) + 1):
                res = self._match_recursive(wildcard_node, tokens, next_idx)
                if res is not None:
                    return res

        return None

    def recommend(self, query: str) -> Tuple[str, bool, float]:
        """
        Recomienda la respuesta emparejada. Retorna (respuesta, es_fallback, tiempo_ms).
        """
        t0 = time.perf_counter()
        tokens = self.normalize_tokens(query)
        if not tokens:
            t_elapsed = (time.perf_counter() - t0) * 1000
            return self.fallback_response, True, t_elapsed

        matched_template = self._match_recursive(self.root, tokens, 0)
        t_elapsed = (time.perf_counter() - t0) * 1000

        if matched_template is not None:
            return matched_template, False, t_elapsed
        else:
            return self.fallback_response, True, t_elapsed

In [ ]:
# --- CONSTRUCCIÓN DEL BANCO DE CATEGORÍAS AIML (DESDE TRAIN) ---
print("[INFO] Compilando árbol Graphmaster AIML desde el dataset de entrenamiento...")
aiml_engine = AIMLGraphmaster(
    fallback_response="No se encontró coincidencia en el motor de patrones sintácticos AIML.")

# Conceptos agronómicos clave para construir plantillas de comodines
AGRI_KEYWORDS = {
    "CACAO", "CAFE", "AGUACATE", "PLATANO", "BANANO", "TOMATE", "MAIZ", "ALGODON",
    "PAPA", "CITRICOS", "NARANJA", "LIMON", "PALMA", "MANGO", "BERENJENA", "FRIJOL", "BATATA",
    "PODA", "FERTILIZACION", "ABONO", "RIEGO", "SEQUIA", "SUELO", "TIERRA", "ENFERMEDAD",
    "HONGO", "PLAGA", "INSECTO", "SIEMBRA", "TRASPLANTE", "COSECHA", "ENCHARCAMIENTO",
    "DRENAJE", "SOMBRIO", "MONILIA", "MACHETE", "BROCA", "ANTRACNOSIS", "CAL", "GALLINAZA"
}

for item in candidate_corpus:
    q_text = item["question"]
    ans_text = item["answer"]

    # 1. Patrón de coincidencia con las preguntas de entrenamiento
    q_tokens = AIMLGraphmaster.normalize_tokens(q_text)
    aiml_engine.add_category(q_tokens, ans_text)

    # 2. Patrones generalizados con comodines basados en palabras clave presentes en la pregunta
    kw_present = [w for w in q_tokens if w in AGRI_KEYWORDS]
    if len(kw_present) >= 2:
        pattern_wildcard = ["*"] + [kw_present[0], "*", kw_present[1], "*"]
        aiml_engine.add_category(pattern_wildcard, ans_text)

print(
    f"✅ Motor AIML compilado exitosamente con {aiml_engine.total_patterns:,} patrones indexados.")

# Prueba de verificación con una pregunta
test_q = "¿Cómo se controla la monilia en cacao?"
sample_ans, is_fb, t_ms = aiml_engine.recommend(test_q)
print(f"\n🧪 Prueba AIML: '{test_q}'")
print(f"   Es Fallback: {is_fb} | Tiempo: {t_ms:.2f} ms")
print(f"   Respuesta: {sample_ans[:100]}...")

### 3.2. Método 2: Modelos Probabilísticos y Vectoriales — Okapi BM25 sobre Respuestas
- **Referencia Clave:** Robertson, S., & Zaragoza, H. (2009). *The Probabilistic Relevance Framework: BM25 and Beyond*. Foundations and Trends in Information Retrieval.
- **Diseño Algorítmico (Question-to-Answer Matching):**
  - Para cumplir con el requerimiento de buscar directamente sobre el catálogo de respuestas prehechas, el índice de BM25 se construye **directamente sobre el texto de las respuestas candidatas** de `dataset_agricola_train.jsonl`.
  - Cuando el agricultor ingresa una pregunta $q$, BM25 puntúa la relevancia de cada respuesta candidata $a_i$:
    $$r^* = \begin{cases} \arg\max_{a_i} \text{Score}_{\text{BM25}}(q, a_i) & \text{si } \max(\text{Score}) \ge \tau \\ r_{\text{fallback}} & \text{en otro caso} \end{cases}$$
  - Parámetros: $k_1=1.5$ (saturación de términos), $b=0.75$ (normalización de longitud de respuesta).
  - Preprocesamiento: Filtrado de puntuación, remoción de stopwords en español y *stemming* de Snowball.


In [ ]:
# --- IMPLEMENTACIÓN DE OKAPI BM25 SOBRE RESPUESTAS CANDIDATAS (ROBERTSON & ZARAGOZA, 2009) ---
from rank_bm25 import BM25Okapi
from nltk.stem.snowball import SpanishStemmer
from nltk.corpus import stopwords
import string


class BM25AnswerRecommender:
    def __init__(self, candidate_items: List[dict], tau: float = 3.0,
                 fallback_response: str = "No se identificó respuesta con suficiente confianza probabilística BM25."):
        self.tau = tau
        self.fallback_response = fallback_response
        self.candidate_items = candidate_items
        self.stemmer = SpanishStemmer()
        self.stop_words = set(stopwords.words('spanish'))

        print(
            "[INFO] Preprocesando y tokenizando catálogo de respuestas para Okapi BM25...")
        # Indexamos directamente sobre el texto de las respuestas candidatas
        self.tokenized_corpus = [self.preprocess(
            item["answer"]) for item in candidate_items]
        self.bm25 = BM25Okapi(self.tokenized_corpus, k1=1.5, b=0.75)
        print(
            f"✅ Índice Okapi BM25 compilado con {len(self.tokenized_corpus):,} respuestas candidatas.")

    def preprocess(self, text: str) -> List[str]:
        """Preprocesamiento en español: minúsculas, filtrado de puntuación, stopwords y stemming."""
        text_clean = text.lower().translate(str.maketrans(
            '', '', string.punctuation + '¿?¡!.,;:()"\'-_\n\r'))
        tokens = text_clean.split()
        stemmed = [self.stemmer.stem(
            t) for t in tokens if t not in self.stop_words and len(t) > 1]
        return stemmed if stemmed else tokens

    def recommend(self, query: str) -> Tuple[str, bool, float, float]:
        """
        Puntúa la pregunta directamente contra todas las respuestas del catálogo.
        Retorna (respuesta_seleccionada, es_fallback, max_score, tiempo_ms).
        """
        t0 = time.perf_counter()
        q_tokens = self.preprocess(query)
        if not q_tokens:
            t_elapsed = (time.perf_counter() - t0) * 1000
            return self.fallback_response, True, 0.0, t_elapsed

        scores = self.bm25.get_scores(q_tokens)
        best_idx = int(scores.argmax())
        max_score = float(scores[best_idx])
        t_elapsed = (time.perf_counter() - t0) * 1000

        if max_score >= self.tau:
            return self.candidate_items[best_idx]["answer"], False, max_score, t_elapsed
        else:
            return self.fallback_response, True, max_score, t_elapsed

In [ ]:
# --- INICIALIZACIÓN DEL RECOMENDADOR BM25 (TAU = 3.0) ---
BM25_TAU = 3.0
bm25_engine = BM25AnswerRecommender(candidate_corpus, tau=BM25_TAU)

# Prueba de verificación
test_q = "Tengo problemas de marchitamiento rápido en las matas de cacao, ¿qué enfermedad puede ser?"
bm25_ans, bm25_fb, bm25_score, bm25_t = bm25_engine.recommend(test_q)
print(f"\n🧪 Prueba BM25 (Question-to-Answer): '{test_q}'")
print(
    f"   Score Relevancia: {bm25_score:.3f} (Umbral tau={BM25_TAU}) | Es Fallback: {bm25_fb} | Tiempo: {bm25_t:.2f} ms")
print(f"   Respuesta Recomendada: {bm25_ans[:120]}...")

### 3.3. Método 3: Razonamiento Basado en Casos Textuales (TCBR) — FAQ Finder Multicanal
- **Referencia Clave:** Burke, R. D., Hammond, K. J., Kulyukin, V., Lytinen, S. L., Tomuro, N., & Schoenberg, S. (1997). *Question Answering from Local Library Resources: An Investigation of FAQ Finder*. User Modeling and User-Adapted Interaction.
- **Diseño Algorítmico Multicanal sobre Respuestas:**
  1. **Canal Léxico ($S_{\text{lex}}$):** Similitud coseno TF-IDF sublineal entre la consulta y el texto de las respuestas candidatas.
  2. **Canal Semántico Ontológico ($S_{\text{sem}}$):** Similitud conceptual sobre el grafo de *WordNet* en español (`omw-1.4`), evaluando la similitud de Wu & Palmer (`wup_similarity`) entre conceptos de la pregunta y de las respuestas:
     $$S_{\text{sem}}(Q, A) = \frac{1}{|Q|} \sum_{w_q \in Q} \max_{w_a \in A} \text{WuPalmer}(w_q, w_a)$$
  3. **Arquitectura en Dos Fases (Burke et al.):**
     - *Fase 1 (Filtro Léxico):* TF-IDF preselecciona las $K=20$ respuestas más prometedoras.
     - *Fase 2 (Reordenamiento Semántico):* Se calcula el canal semántico WordNet sobre los Top-20 candidatos.
     - Puntuación combinada:
       $$S_{\text{TCBR}}(q, a_i) = \alpha \cdot S_{\text{lex}}(q, a_i) + (1 - \alpha) \cdot S_{\text{sem}}(q, a_i)$$


In [ ]:
# --- IMPLEMENTACIÓN DE FAQ FINDER TCBR SOBRE RESPUESTAS CANDIDATAS (BURKE ET AL., 1997) ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import wordnet as wn


class FAQFinderTCBR:
    def __init__(self, candidate_items: List[dict], alpha: float = 0.5, top_k: int = 20, tau: float = 0.15,
                 fallback_response: str = "No se identificó respuesta con suficiente similitud en FAQ Finder."):
        self.candidate_items = candidate_items
        self.alpha = alpha
        self.top_k = top_k
        self.tau = tau
        self.fallback_response = fallback_response
        self.stop_words = set(stopwords.words('spanish'))

        print(
            "[INFO] Ajustando vectorizador TF-IDF para Canal Léxico sobre Respuestas...")
        self.vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            sublinear_tf=True,
            stop_words=list(self.stop_words)
        )
        self.corpus_answers = [item["answer"] for item in candidate_items]
        self.tfidf_matrix = self.vectorizer.fit_transform(self.corpus_answers)

        print("[INFO] Pre-extrayendo conceptos semánticos del catálogo para WordNet...")
        self.corpus_content_words = [
            self._extract_content_words(a) for a in self.corpus_answers]
        print(
            f"✅ FAQ Finder TCBR inicializado con {len(candidate_items):,} respuestas indexadas.")

    def _extract_content_words(self, text: str) -> List[str]:
        """Extrae palabras de contenido con longitud relevante omitiendo stopwords."""
        clean = re.sub(r'[^a-zA-ZáéíóúÁÉÍÓÚñÑüÜ\s]', ' ', text.lower())
        tokens = [t for t in clean.split(
        ) if t not in self.stop_words and len(t) > 2]
        # Limitar a las 50 palabras más representativas para eficiencia
        return tokens[:50]

    def _wordnet_similarity(self, words_q: List[str], words_a: List[str]) -> float:
        """Calcula la similitud semántica media Wu-Palmer sobre WordNet en español."""
        if not words_q or not words_a:
            return 0.0

        scores = []
        for wq in words_q:
            syns_q = wn.synsets(wq, lang='spa')
            if not syns_q:
                continue

            max_sim = 0.0
            for wa in words_a:
                if wq == wa:
                    max_sim = 1.0
                    break
                syns_a = wn.synsets(wa, lang='spa')
                if not syns_a:
                    continue

                for sq in syns_q:
                    for sa in syns_a:
                        sim = sq.wup_similarity(sa)
                        if sim and sim > max_sim:
                            max_sim = sim
            scores.append(max_sim)

        return float(sum(scores) / len(scores)) if scores else 0.0

    def recommend(self, query: str) -> Tuple[str, bool, float, float]:
        """
        Ejecuta el pipeline en dos etapas de FAQ Finder sobre las respuestas candidatas.
        Retorna (respuesta_seleccionada, es_fallback, score_combinado, tiempo_ms).
        """
        t0 = time.perf_counter()
        query_words = self._extract_content_words(query)

        # Etapa 1: Filtro Léxico Rápido (TF-IDF Cosine Similarity)
        q_vec = self.vectorizer.transform([query])
        lex_sims = cosine_similarity(q_vec, self.tfidf_matrix).flatten()

        top_indices = lex_sims.argsort()[-self.top_k:][::-1]

        # Etapa 2: Ponderación Semántica con WordNet sobre candidatos Top-K
        best_score = -1.0
        best_candidate_idx = -1

        for idx in top_indices:
            s_lex = float(lex_sims[idx])
            s_sem = self._wordnet_similarity(
                query_words, self.corpus_content_words[idx])
            s_total = (self.alpha * s_lex) + ((1.0 - self.alpha) * s_sem)

            if s_total > best_score:
                best_score = s_total
                best_candidate_idx = idx

        t_elapsed = (time.perf_counter() - t0) * 1000

        if best_score >= self.tau and best_candidate_idx >= 0:
            return self.candidate_items[best_candidate_idx]["answer"], False, best_score, t_elapsed
        else:
            return self.fallback_response, True, best_score, t_elapsed

In [ ]:
# --- INICIALIZACIÓN DE FAQ FINDER TCBR (TAU = 0.15) ---
faq_engine = FAQFinderTCBR(candidate_corpus, alpha=0.5, top_k=20, tau=0.15)

# Prueba de verificación
test_q = "¿Qué árboles dan buena sombra para el cacao y no le compiten con las raíces?"
faq_ans, faq_fb, faq_score, faq_t = faq_engine.recommend(test_q)
print(f"\n🧪 Prueba FAQ Finder (TCBR): '{test_q}'")
print(
    f"   Score Multicanal: {faq_score:.3f} | Es Fallback: {faq_fb} | Tiempo: {faq_t:.2f} ms")
print(f"   Respuesta: {faq_ans[:120]}...")

### 3.4. Método 4: Motores Simbólicos y Reglas — Sistema Experto con Inferencia Rete
- **Referencia Clave:** Forgy, C. L. (1982). *Rete: A Fast Algorithm for the Many Pattern/Many Object Pattern Match Problem*. Artificial Intelligence; Felfernig, A., et al. (2006). *Constraint-based recommender systems*.
- **Diseño Algorítmico (Inferencia Hacia Adelante / Forward Chaining):**
  - La consulta se analiza mediante un extractor de hechos (*Working Memory*): cultivo, temática agronómica y síntoma/condición específica.
  - **Red Rete:**
    - **Nodos Alfa:** Filtros unarios sobre atributos de hechos (ej. `Crop == 'cacao'`).
    - **Nodos Beta / Join Nodes:** Realizan conjunciones lógicas entre múltiples hechos (ej. `Crop == 'cacao' AND Topic == 'poda'`).
    - **Nodos Terminales:** Reglas de producción compiladas **únicamente desde el conjunto de entrenamiento** que asocian las condiciones con la mejor respuesta experta.
  - **Resolución de Conflictos:** Mayor especificidad (*Highest Specificity*): las reglas que coinciden con más condiciones tienen prelación sobre reglas generales.


In [ ]:
# --- IMPLEMENTACIÓN DE RED RETE NATIVA PARA SISTEMA EXPERTO AGRÍCOLA (FORGY, 1982) ---

Fact = namedtuple('Fact', ['type', 'value'])


class AlphaNode:
    """Nodo Alfa: Filtro unario de hechos en la Red Rete."""

    def __init__(self, fact_type: str, fact_value: str):
        self.fact_type = fact_type
        self.fact_value = fact_value
        self.memory: List[Fact] = []
        self.successors: List['BetaNode'] = []

    def activate(self, fact: Fact):
        if fact.type == self.fact_type and fact.value == self.fact_value:
            if fact not in self.memory:
                self.memory.append(fact)
                for succ in self.successors:
                    succ.right_activate(fact)

    def clear(self):
        self.memory.clear()


class BetaNode:
    """Nodo Beta / Join Node: Conjunción binaria de condiciones lógicas."""

    def __init__(self, parent_beta: Optional['BetaNode'], alpha_node: AlphaNode):
        self.parent_beta = parent_beta
        self.alpha_node = alpha_node
        self.memory: List[Tuple[Fact, ...]] = []
        self.terminal_nodes: List['TerminalRuleNode'] = []
        alpha_node.successors.append(self)

    def right_activate(self, fact: Fact):
        if self.parent_beta is None:
            token = (fact,)
            if token not in self.memory:
                self.memory.append(token)
                for term in self.terminal_nodes:
                    term.activate(token)
        else:
            for parent_token in self.parent_beta.memory:
                new_token = parent_token + (fact,)
                if new_token not in self.memory:
                    self.memory.append(new_token)
                    for term in self.terminal_nodes:
                        term.activate(new_token)

    def clear(self):
        self.memory.clear()


class TerminalRuleNode:
    """Nodo Terminal: Regla de producción activada que recomienda una respuesta."""

    def __init__(self, rule_id: str, conditions_count: int, answer_text: str):
        self.rule_id = rule_id
        self.specificity = conditions_count
        self.answer_text = answer_text
        self.activations: List[Tuple[Fact, ...]] = []

    def activate(self, token: Tuple[Fact, ...]):
        self.activations.append(token)

    def clear(self):
        self.activations.clear()


class ReteExpertSystem:
    """Motor de Sistema Experto de Producción con Red Rete para Recomendación Agrícola."""

    def __init__(self, fallback_response: str = "Ninguna regla experta Rete se activó para los hechos identificados."):
        self.alpha_nodes: Dict[Tuple[str, str], AlphaNode] = {}
        self.terminal_nodes: List[TerminalRuleNode] = []
        self.all_beta_nodes: List[BetaNode] = []
        self.fallback_response = fallback_response
        self._build_fact_taxonomies()

    def _build_fact_taxonomies(self):
        """Define taxonomía de entidades para extraer hechos sintácticos de la consulta."""
        self.crops = {
            "cacao": ["cacao", "cacaotero", "theobroma"],
            "cafe": ["cafe", "cafeto", "café"],
            "aguacate": ["aguacate", "palta", "hass"],
            "platano": ["platano", "plátano", "banano", "musáceas"],
            "tomate": ["tomate", "tomates", "solanum"],
            "maiz": ["maiz", "maíz", "choclo"],
            "algodon": ["algodon", "algodón", "algodonero"],
            "papa": ["papa", "papas", "patata", "criolla"],
            "citricos": ["citricos", "cítricos", "naranja", "limon", "limón", "tahití"],
            "berenjena": ["berenjena", "berenjenas"],
            "caucho": ["caucho", "hevea"],
            "batata": ["batata", "camote"],
            "mora": ["mora", "moras"]
        }

        self.topics = {
            "poda": ["poda", "podar", "deschuponar", "chupones", "horqueta", "ramas"],
            "fertilizacion": ["fertilizacion", "fertilizar", "abono", "abonar", "gallinaza", "nutrientes", "compost", "cal"],
            "riego_sequia": ["riego", "regar", "sequia", "sequía", "verano", "agua", "deficit hidrico"],
            "suelo": ["suelo", "tierra", "analisis de suelo", "acidez", "ph", "materia organica"],
            "drenaje_encharque": ["encharcamiento", "encharcar", "drenaje", "inundacion", "agua estancada", "barro"],
            "plagas": ["plaga", "plagas", "bichos", "chinche", "gusano", "oruga", "insecto", "larva", "mosca blanca"],
            "enfermedades": ["enfermedad", "enfermedades", "hongo", "monilia", "machete", "marchitamiento", "antracnosis", "podrido", "manchas"],
            "siembra_trasplante": ["siembra", "sembrar", "trasplante", "trasplantar", "semillero", "vivero", "almacigo"],
            "sombrio": ["sombrio", "sombrío", "sombra", "guamo", "arboles de sombra"]
        }

    def _get_or_create_alpha(self, f_type: str, f_val: str) -> AlphaNode:
        key = (f_type, f_val)
        if key not in self.alpha_nodes:
            self.alpha_nodes[key] = AlphaNode(f_type, f_val)
        return self.alpha_nodes[key]

    def add_rule(self, rule_id: str, conditions: List[Tuple[str, str]], answer_text: str):
        """Compila una regla en la Red Rete creando nodos Alfa, Beta y Terminal."""
        if not conditions:
            return

        current_beta = None
        for i, (f_type, f_val) in enumerate(conditions):
            alpha = self._get_or_create_alpha(f_type, f_val)
            new_beta = BetaNode(parent_beta=current_beta, alpha_node=alpha)
            self.all_beta_nodes.append(new_beta)
            current_beta = new_beta

        terminal = TerminalRuleNode(rule_id=rule_id, conditions_count=len(
            conditions), answer_text=answer_text)
        current_beta.terminal_nodes.append(terminal)
        self.terminal_nodes.append(terminal)

    def extract_facts(self, query: str) -> List[Fact]:
        """Extrae hechos sintácticos de la consulta en lenguaje natural."""
        q_lower = query.lower()
        facts = []

        for crop_id, aliases in self.crops.items():
            if any(re.search(r'\b' + re.escape(a) + r'\b', q_lower) for a in aliases):
                facts.append(Fact("crop", crop_id))

        for topic_id, aliases in self.topics.items():
            if any(re.search(r'\b' + re.escape(a) + r'\b', q_lower) for a in aliases):
                facts.append(Fact("topic", topic_id))

        return facts

    def recommend(self, query: str) -> Tuple[str, bool, Optional[str], float]:
        """
        Propaga hechos en la Red Rete y dispara la regla con mayor especificidad.
        Retorna (respuesta, es_fallback, rule_id_disparada, tiempo_ms).
        """
        t0 = time.perf_counter()

        for alpha in self.alpha_nodes.values():
            alpha.clear()
        for beta in self.all_beta_nodes:
            beta.clear()
        for term in self.terminal_nodes:
            term.clear()

        facts = self.extract_facts(query)
        for fact in facts:
            if (fact.type, fact.value) in self.alpha_nodes:
                self.alpha_nodes[(fact.type, fact.value)].activate(fact)

        active_rules = [
            term for term in self.terminal_nodes if len(term.activations) > 0]
        t_elapsed = (time.perf_counter() - t0) * 1000

        if active_rules:
            active_rules.sort(key=lambda r: r.specificity, reverse=True)
            best_rule = active_rules[0]
            return best_rule.answer_text, False, best_rule.rule_id, t_elapsed
        else:
            return self.fallback_response, True, None, t_elapsed

In [ ]:
# --- COMPILACIÓN DE REGLAS EXPERTAS (DESDE TRAIN) ---
print("[INFO] Compilando base de reglas de producción para la Red Rete desde entrenamiento...")
rete_engine = ReteExpertSystem()

rule_counter = 0
seen_rule_signatures = set()

for item in candidate_corpus:
    facts = rete_engine.extract_facts(item["question"])
    crop_facts = [f for f in facts if f.type == "crop"]
    topic_facts = [f for f in facts if f.type == "topic"]

    if crop_facts and topic_facts:
        c = crop_facts[0]
        t = topic_facts[0]
        sig = (c.value, t.value)

        if sig not in seen_rule_signatures:
            seen_rule_signatures.add(sig)
            rule_counter += 1
            rule_id = f"RULE_{c.value.upper()}_{t.value.upper()}_{rule_counter}"
            conditions = [(c.type, c.value), (t.type, t.value)]
            rete_engine.add_rule(rule_id, conditions, item["answer"])

print(
    f"✅ Red Rete compilada con {rule_counter} reglas expertas y {len(rete_engine.alpha_nodes)} nodos alfa.")

# Prueba de verificación
test_q = "¿Qué fertilizante o abono se debe aplicar en el cultivo de tomate?"
rete_ans, rete_fb, rete_rule, rete_t = rete_engine.recommend(test_q)
print(f"\n🧪 Prueba Sistema Experto Rete: '{test_q}'")
print(
    f"   Regla Disparada: {rete_rule} | Es Fallback: {rete_fb} | Tiempo: {rete_t:.2f} ms")
print(f"   Respuesta Recomendada: {rete_ans[:120]}...")

## 4. Configuración del Evaluador Semántico BERTScore

Siguiendo estrictamente la metodología de evaluación de la tesis:
- **Métrica:** BERTScore Multilingüe en Español.
- **Configuración:** `lang='es'`, `rescale_with_baseline=False` para comparación directa con el sistema RAG generativo de la tesis.
- **Dispositivo:** Aceleración por GPU CUDA.


In [ ]:
import bert_score
from bert_score import BERTScorer

print("[INFO] Inicializando evaluador BERTScore multilingüe (español)...")
bert_scorer = BERTScorer(
    lang='es',
    device='cuda' if torch.cuda.is_available() else 'cpu',
    rescale_with_baseline=False
)
print("✅ BERTScore inicializado exitosamente.")

# Verificación de sanity check
P_test, R_test, F1_test = bert_scorer.score(
    ["El cacao requiere poda al inicio de lluvias."],
    ["Se recomienda podar las plantas de cacao cuando empieza a llover."]
)
print(
    f"🧪 Sanity Check BERTScore: Precision={P_test[0].item():.4f}, Recall={R_test[0].item():.4f}, F1={F1_test[0].item():.4f}")

## 5. Bucle de Evaluación Comparativa Unificada sobre el Dataset Held-Out

En esta etapa:
1. Iteramos sobre las preguntas de consulta de `dataset_agricola_eval.jsonl` (desconocidas para los modelos).
2. Consultamos cada uno de los 4 sistemas tradicionales registrando la respuesta seleccionada y el tiempo de respuesta.
3. Evaluamos mediante **BERTScore** por lotes (*batches*) comparando la respuesta recomendada por cada método frente a la respuesta experta de referencia del dataset de evaluación.
4. Implementamos un sistema de **Checkpointing Resiliente** en `traditional_kbr_checkpoint.jsonl` para prevenir pérdida de datos ante desconexiones de Google Colab.


In [ ]:
import os
import json
import time
import pandas as pd
from tqdm.auto import tqdm

# --- CONFIGURACIÓN DE LA EVALUACIÓN ---
# EVAL_MAX_SAMPLES: Número de muestras a evaluar. Usa None para evaluar el dataset completo (~876 preguntas).
# Puedes usar un entero (ej. 100) para una ejecución preliminar rápida.
EVAL_MAX_SAMPLES = None

CHECKPOINT_FILE = "traditional_kbr_checkpoint.jsonl"

processed_ids = set()
evaluated_results = []

if os.path.exists(CHECKPOINT_FILE):
    print(
        f"[INFO] Se detectó checkpoint existente: '{CHECKPOINT_FILE}'. Leyendo registros previos...")
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    rec = json.loads(line)
                    evaluated_results.append(rec)
                    processed_ids.add(rec["sample_id"])
                except Exception:
                    continue
    print(
        f"✅ Reanudando evaluación: {len(evaluated_results)} muestras previamente completadas.")

samples_to_eval = eval_dataset if EVAL_MAX_SAMPLES is None else eval_dataset[
    :EVAL_MAX_SAMPLES]
pending_samples = [s for s in samples_to_eval if s["id"] not in processed_ids]
print(
    f"[INFO] Muestras pendientes por evaluar: {len(pending_samples):,} de {len(samples_to_eval):,}.")

In [ ]:
# --- BUCLE DE CONSULTA A LOS 4 SISTEMAS Y RECOLECCIÓN DE RESPUESTAS ---

if pending_samples:
    print(
        f"\n🚀 Iniciando recolección de recomendaciones sobre {len(pending_samples)} consultas held-out...")
    batch_records = []

    for sample in tqdm(pending_samples, desc="Consultando KBRs"):
        s_id = sample["id"]
        q_text = sample["question"]
        ref_ans = sample["answer"]

        # 1. AIML
        ans_aiml, fb_aiml, t_aiml = aiml_engine.recommend(q_text)

        # 2. Okapi BM25 sobre Respuestas
        ans_bm25, fb_bm25, score_bm25, t_bm25 = bm25_engine.recommend(q_text)

        # 3. FAQ Finder (TCBR) sobre Respuestas
        ans_faq, fb_faq, score_faq, t_faq = faq_engine.recommend(q_text)

        # 4. Sistema Experto Rete
        ans_rete, fb_rete, rule_rete, t_rete = rete_engine.recommend(q_text)

        record = {
            "sample_id": s_id,
            "question": q_text,
            "reference_answer": ref_ans,
            # Respuestas seleccionadas
            "ans_aiml": ans_aiml,
            "ans_bm25": ans_bm25,
            "ans_faq": ans_faq,
            "ans_rete": ans_rete,
            # Indicadores de fallback
            "fb_aiml": fb_aiml,
            "fb_bm25": fb_bm25,
            "fb_faq": fb_faq,
            "fb_rete": fb_rete,
            # Tiempos de ejecución (ms)
            "time_aiml_ms": t_aiml,
            "time_bm25_ms": t_bm25,
            "time_faq_ms": t_faq,
            "time_rete_ms": t_rete,
            # Metadatos
            "bm25_score": score_bm25,
            "faq_score": score_faq,
            "rete_rule": rule_rete
        }
        batch_records.append(record)

    # --- EVALUACIÓN VECTORIAL BERTSCORE POR LOTES (GPU BATCHING) ---
    print("\n[INFO] Calculando BERTScore para las respuestas seleccionadas por cada método frente a la referencia...")

    refs = [r["reference_answer"] for r in batch_records]

    # Evaluar AIML
    print("  -> Evaluando AIML...")
    cands_aiml = [r["ans_aiml"] for r in batch_records]
    P_a, R_a, F1_a = bert_scorer.score(cands_aiml, refs, batch_size=64)

    # Evaluar BM25
    print("  -> Evaluando Okapi BM25...")
    cands_bm25 = [r["ans_bm25"] for r in batch_records]
    P_b, R_b, F1_b = bert_scorer.score(cands_bm25, refs, batch_size=64)

    # Evaluar FAQ Finder
    print("  -> Evaluando FAQ Finder (TCBR)...")
    cands_faq = [r["ans_faq"] for r in batch_records]
    P_f, R_f, F1_f = bert_scorer.score(cands_faq, refs, batch_size=64)

    # Evaluar Rete
    print("  -> Evaluando Sistema Experto Rete...")
    cands_rete = [r["ans_rete"] for r in batch_records]
    P_r, R_r, F1_r = bert_scorer.score(cands_rete, refs, batch_size=64)

    # Integrar métricas a los registros y persistir checkpoint
    with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f_ckpt:
        for idx, r in enumerate(batch_records):
            r["bertscore_aiml_p"] = float(P_a[idx].item()) if not r["fb_aiml"] else 0.0
            r["bertscore_aiml_r"] = float(R_a[idx].item()) if not r["fb_aiml"] else 0.0
            r["bertscore_aiml_f1"] = float(F1_a[idx].item()) if not r["fb_aiml"] else 0.0

            r["bertscore_bm25_p"] = float(P_b[idx].item()) if not r["fb_bm25"] else 0.0
            r["bertscore_bm25_r"] = float(R_b[idx].item()) if not r["fb_bm25"] else 0.0
            r["bertscore_bm25_f1"] = float(F1_b[idx].item()) if not r["fb_bm25"] else 0.0

            r["bertscore_faq_p"] = float(P_f[idx].item()) if not r["fb_faq"] else 0.0
            r["bertscore_faq_r"] = float(R_f[idx].item()) if not r["fb_faq"] else 0.0
            r["bertscore_faq_f1"] = float(F1_f[idx].item()) if not r["fb_faq"] else 0.0

            r["bertscore_rete_p"] = float(P_r[idx].item()) if not r["fb_rete"] else 0.0
            r["bertscore_rete_r"] = float(R_r[idx].item()) if not r["fb_rete"] else 0.0
            r["bertscore_rete_f1"] = float(F1_r[idx].item()) if not r["fb_rete"] else 0.0

            f_ckpt.write(json.dumps(r, ensure_ascii=False) + "\n")
            evaluated_results.append(r)

    print(f"✅ Evaluación completada y guardada en '{CHECKPOINT_FILE}'.")
else:
    print("✅ Todas las muestras seleccionadas ya estaban evaluadas en el checkpoint.")

## 6. Análisis Estadístico y Tabla Comparativa Consolidada

A continuación calculamos los estadísticos descriptivos (Media $\mu$ y Desviación Estándar $\sigma$) para cada dimensión:
- **BERTScore Precision, Recall y F1-Score**.
- **Tasa de Fallback (% de consultas sin respuesta específica encontrada)** y **Cobertura (%)**.
- **Latencia promedio de inferencia (ms)**.

Para contrastar con la solución desarrollada en la tesis, incluimos los **valores empíricos reales del Sistema RAG FarmifAI (SLM Qwen3.5 + Hybrid Reranker)** obtenidos en la evaluación oficial de la tesis.


In [ ]:
import pandas as pd
import numpy as np

# Cargar todos los resultados evaluados en un DataFrame
df_results = pd.DataFrame(evaluated_results)
print(f"[INFO] Consultas totales evaluadas: {len(df_results)}")

# --- CONSTRUCCIÓN DEL SCORECARD CONSOLIDADO ---
methods_config = [
    {
        "Método": "AIML (Wallace, 2009)",
        "Paradigma": "Sintáctico / Árbol Graphmaster",
        "p_col": "bertscore_aiml_p",
        "r_col": "bertscore_aiml_r",
        "f1_col": "bertscore_aiml_f1",
        "fb_col": "fb_aiml",
        "t_col": "time_aiml_ms"
    },
    {
        "Método": "Okapi BM25 (τ=3.0)",
        "Paradigma": "Probabilístico / IR sobre Respuestas",
        "p_col": "bertscore_bm25_p",
        "r_col": "bertscore_bm25_r",
        "f1_col": "bertscore_bm25_f1",
        "fb_col": "fb_bm25",
        "t_col": "time_bm25_ms"
    },
    {
        "Método": "FAQ Finder (Burke et al., 1997)",
        "Paradigma": "TCBR / Léxico + WordNet en Respuestas",
        "p_col": "bertscore_faq_p",
        "r_col": "bertscore_faq_r",
        "f1_col": "bertscore_faq_f1",
        "fb_col": "fb_faq",
        "t_col": "time_faq_ms"
    },
    {
        "Método": "Sistema Experto Rete (Forgy, 1982)",
        "Paradigma": "Simbólico / Forward Chaining",
        "p_col": "bertscore_rete_p",
        "r_col": "bertscore_rete_r",
        "f1_col": "bertscore_rete_f1",
        "fb_col": "fb_rete",
        "t_col": "time_rete_ms"
    }
]

summary_rows = []
for m in methods_config:
    p_mean = df_results[m["p_col"]].mean()
    p_std = df_results[m["p_col"]].std()

    r_mean = df_results[m["r_col"]].mean()
    r_std = df_results[m["r_col"]].std()

    f1_mean = df_results[m["f1_col"]].mean()
    f1_std = df_results[m["f1_col"]].std()

    fb_rate = (df_results[m["fb_col"]].sum() / len(df_results)) * 100
    cov_rate = 100.0 - fb_rate
    t_mean = df_results[m["t_col"]].mean()

    summary_rows.append({
        "Método de Recomendación": m["Método"],
        "Paradigma Algorítmico": m["Paradigma"],
        "BERTScore Precision": f"{p_mean:.3f} ± {p_std:.3f}",
        "BERTScore Recall": f"{r_mean:.3f} ± {r_std:.3f}",
        "BERTScore F1": f"{f1_mean:.3f} ± {f1_std:.3f}",
        "Cobertura (%)": f"{cov_rate:.1f}%",
        "Fallback (%)": f"{fb_rate:.1f}%",
        "Latencia Media": f"{t_mean:.2f} ms",
        # Métricas crudas para gráficos
        "raw_p": p_mean,
        "raw_r": r_mean,
        "raw_f1": f1_mean,
        "raw_coverage": cov_rate,
        "raw_latency_ms": t_mean
    })

# Fila comparativa con los RESULTADOS REALES EMPÍRICOS DE FARMIFI (Tesis Oficial)
summary_rows.append({
    "Método de Recomendación": "FarmifAI RAG (SLM Qwen3.5 + Hybrid Reranker) 🌟",
    "Paradigma Algorítmico": "Generativo / RAG Híbrido End-to-End",
    "BERTScore Precision": "0.729 ± 0.035",
    "BERTScore Recall": "0.731 ± 0.033",
    "BERTScore F1": "0.730 ± 0.031",
    "Cobertura (%)": "100.0%",
    "Fallback (%)": "0.0%",
    "Latencia Media": "85,874.00 ms (85.87 s)",
    "raw_p": 0.729,
    "raw_r": 0.731,
    "raw_f1": 0.730,
    "raw_coverage": 100.0,
    "raw_latency_ms": 85874.0
})

df_summary = pd.DataFrame(summary_rows)
display_cols = ["Método de Recomendación", "Paradigma Algorítmico", "BERTScore Precision",
                "BERTScore Recall", "BERTScore F1", "Cobertura (%)", "Fallback (%)", "Latencia Media"]

print("\n" + "=" * 110)
print(" 📋 TABLA COMPARATIVA CONSOLIDADA: KBRs TRADICIONALES vs. RESULTADOS REALES FARMIFI RAG")
print("=" * 110)
print(df_summary[display_cols].to_string(index=False))
print("=" * 110)

print("""
📌 DATOS COMPLEMENTARIOS DE EVALUACIÓN OFICIAL DE FARMIFI (LLM as a Judge & G-Eval):
  • Context Relevancy (RAG Triad): 3.700 ± 1.106  [Bueno]
  • Faithfulness (RAG Triad):      3.096 ± 1.212  [Regular]
  • Answer Relevancy (RAG Triad):  3.488 ± 1.134  [Regular]
  • G-Eval Coherencia:             3.672 ± 0.976  [Bueno]
  • G-Eval Consistencia:           3.376 ± 1.240  [Regular]
  • G-Eval Fluidez:                4.236 ± 0.809  [Bueno]
  • Latencia Recuperación RAG:     0.316 ± 0.145 s (316 ms)
  • Latencia Generación SLM:       85.558 ± 14.718 s
  • Velocidad Generación:          4.683 ± 0.278 tok/s
""")

## 7. Dashboard Gráfico de Evaluación Comparativa

1. **Barras Comparativas de Calidad Semántica (BERTScore P, R, F1)** frente al baseline FarmifAI RAG ($F_1 = 0.730$).
2. **Diagramas de Caja (Boxplots)** que evidencian la variabilidad y estabilidad de los métodos tradicionales frente a consultas no estructuradas.
3. **Compromiso de Eficiencia y Latencia (Escala Logarítmica):** Comparativa del tiempo de respuesta entre KBRs tradicionales (milisegundos) y la generación generativa local con SLM en Colab (~85.8 s).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.style.use(
    'seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig_dpi = 300

# --- FIGURA 1: BARRAS COMPARATIVAS DE BERTSCORE (P, R, F1) ---
fig, ax = plt.subplots(figsize=(12, 6), dpi=fig_dpi)

methods_labels = [r["Método de Recomendación"].split(
    "(")[0].replace("🌟", "").strip() for r in summary_rows]
x = np.arange(len(methods_labels))
width = 0.25

p_vals = [r["raw_p"] for r in summary_rows]
r_vals = [r["raw_r"] for r in summary_rows]
f1_vals = [r["raw_f1"] for r in summary_rows]

rects1 = ax.bar(x - width, p_vals, width, label='Precision',
                color='#1f77b4', edgecolor='black', alpha=0.85)
rects2 = ax.bar(x, r_vals, width, label='Recall',
                color='#2ca02c', edgecolor='black', alpha=0.85)
rects3 = ax.bar(x + width, f1_vals, width, label='F1-Score',
                color='#d62728', edgecolor='black', alpha=0.9)

ax.set_ylabel('BERTScore (Español)', fontsize=12, fontweight='bold')
ax.set_title('Comparativa de Rendimiento Semántico: Sistemas KBR Tradicionales vs. FarmifAI RAG Real',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(methods_labels, rotation=15, ha='right', fontsize=11)
ax.set_ylim(0, 1.05)
ax.axhline(0.730, color='#d62728', linestyle='--', linewidth=1.8,
           alpha=0.7, label='FarmifAI RAG Baseline F1 (0.730)')
ax.legend(frameon=True, facecolor='white', framealpha=0.9, fontsize=11)

for rect in rects3:
    height = rect.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig("kbr_metrics_comparison.png", dpi=fig_dpi)
plt.show()

In [ ]:
# --- FIGURA 2: DISTRIBUCIÓN Y DISPERSIÓN DE F1-SCORE (BOXPLOTS) ---
fig, ax = plt.subplots(figsize=(10, 5), dpi=fig_dpi)

plot_data = [
    df_results["bertscore_aiml_f1"].dropna(),
    df_results["bertscore_bm25_f1"].dropna(),
    df_results["bertscore_faq_f1"].dropna(),
    df_results["bertscore_rete_f1"].dropna()
]
labels_box = ["AIML", "Okapi BM25", "FAQ Finder", "Sistema Experto"]

box = ax.boxplot(plot_data, labels=labels_box, patch_artist=True,
                 boxprops=dict(facecolor='#bde0fe',
                               color='#1d3557', alpha=0.8),
                 medianprops=dict(color='#e63946', linewidth=2.2),
                 flierprops=dict(marker='o', markersize=3, alpha=0.3))

ax.set_title('Dispersión de BERTScore F1 en Sistemas KBR Tradicionales',
             fontsize=13, fontweight='bold')
ax.set_ylabel('BERTScore F1', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.axhline(0.730, color='#d62728', linestyle='--', linewidth=1.5,
           alpha=0.6, label='FarmifAI RAG F1 (0.730)')
ax.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.savefig("kbr_f1_distribution.png", dpi=fig_dpi)
plt.show()

# --- FIGURA 3: COMPARATIVA DE LATENCIA (ESCALA LOGARÍTMICA MS) ---
fig, ax = plt.subplots(figsize=(11, 4.5), dpi=fig_dpi)
latencies_ms = [r["raw_latency_ms"] for r in summary_rows]
colors_lat = ['#457b9d', '#457b9d', '#457b9d', '#457b9d', '#e63946']

bars = ax.barh(methods_labels, latencies_ms, color=colors_lat,
               edgecolor='black', alpha=0.85)
ax.set_xscale('log')
ax.set_xlabel('Latencia Media por Consulta (milisegundos, escala logarítmica)',
              fontsize=11, fontweight='bold')
ax.set_title('Compromiso de Eficiencia: Latencia de Inferencia por Paradigma',
             fontsize=13, fontweight='bold')

for bar in bars:
    w = bar.get_width()
    text_label = f'{w:.1f} ms' if w < 1000 else f'{w/1000:.2f} s ({w:,.0f} ms)'
    ax.annotate(text_label,
                xy=(w, bar.get_y() + bar.get_height() / 2),
                xytext=(6, 0), textcoords="offset points",
                ha='left', va='center', fontsize=9.5, fontweight='bold')

plt.tight_layout()
plt.savefig("kbr_latency_comparison.png", dpi=fig_dpi)
plt.show()

## 8. Exportación de Resultados para la Memoria de Tesis

Guardamos los resultados detallados a nivel de consulta individual y la tabla resumen en formato CSV compatible con LaTeX y Excel. Asimismo, se descargan automáticamente los gráficos generados.


In [ ]:
# --- EXPORTACIÓN DE RESULTADOS A ARCHIVOS CSV ---

RESULTS_FULL_CSV = "traditional_kbr_full_results.csv"
RESULTS_SUMMARY_CSV = "traditional_kbr_summary_metrics.csv"

# 1. Guardar resultados muestra por muestra
df_results.to_csv(RESULTS_FULL_CSV, index=False, encoding="utf-8-sig")
print(
    f"✅ Archivo detallado exportado: '{RESULTS_FULL_CSV}' ({len(df_results)} registros)")

# 2. Guardar tabla resumen consolidada
df_summary[display_cols].to_csv(
    RESULTS_SUMMARY_CSV, index=False, encoding="utf-8-sig")
print(f"✅ Resumen estadístico exportado: '{RESULTS_SUMMARY_CSV}'")

# Descarga en Google Colab
try:
    from google.colab import files
    print("\n[INFO] Descargando reportes y figuras a tu equipo local...")
    files.download(RESULTS_SUMMARY_CSV)
    files.download(RESULTS_FULL_CSV)
    files.download("kbr_metrics_comparison.png")
    files.download("kbr_f1_distribution.png")
    files.download("kbr_latency_comparison.png")
    print("✅ Descarga iniciada.")
except Exception as e:
    print(f"ℹ️ Archivos listos en el directorio local de Colab: {e}")